In [2]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import evaluate
import matplotlib.pyplot as plt
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch

nltk.download("punkt")
nltk.download("punkt_tab")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\MSI\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\MSI\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [1]:
from huggingface_hub import login
login("*********************") #Enter api key here

C:\Users\MSI\anaconda3\envs\GEN_AI_ENV\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [4]:
model_ckpt = "SEBIS/code_trans_t5_small_api_generation"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

In [5]:
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

Loading weights: 100%|██████████| 133/133 [00:00<00:00, 16625.21it/s]


In [6]:
dataset_samsum = load_dataset("knkarthick/samsum")
dataset_samsum

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

In [7]:
dataset_samsum["train"]["dialogue"][1]

'Olivia: Who are you voting for in this election? \nOliver: Liberals as always.\nOlivia: Me too!!\nOliver: Great'

In [8]:
dataset_samsum["train"]["summary"][1]

'Olivia and Olivier are voting for liberals in this election. '

In [9]:
split_lengths = [len(dataset_samsum[split]) for split in dataset_samsum]

print(f"Split lengths: {split_lengths}")
print(f"Features: {dataset_samsum['train'].column_names}")
print("\nDialogue:")

print(dataset_samsum["test"][1]["dialogue"])
print(f"\nSummary: ")

print(dataset_samsum["test"][1]["summary"])

Split lengths: [14731, 818, 819]
Features: ['id', 'dialogue', 'summary']

Dialogue:
Eric: MACHINE!
Rob: That's so gr8!
Eric: I know! And shows how Americans see Russian ;)
Rob: And it's really funny!
Eric: I know! I especially like the train part!
Rob: Hahaha! No one talks to the machine like that!
Eric: Is this his only stand-up?
Rob: Idk. I'll check.
Eric: Sure.
Rob: Turns out no! There are some of his stand-ups on youtube.
Eric: Gr8! I'll watch them now!
Rob: Me too!
Eric: MACHINE!
Rob: MACHINE!
Eric: TTYL?
Rob: Sure :)

Summary: 
Eric and Rob are going to watch a stand-up on youtube.


In [10]:
def convert_examples_to_features(example_batch):
  input_encodings = tokenizer(example_batch["dialogue"], max_length=128, truncation=True)


#with tokenizer.as_target_tokenizer is old one so removed

  target_encodings = tokenizer(example_batch["summary"], max_length=32, truncation=True)

  return {
      "input_ids": input_encodings["input_ids"],
      "attention_mask": input_encodings["attention_mask"],
      "labels": target_encodings["input_ids"]
  }

In [11]:
dataset_samsum_pt = dataset_samsum.map(convert_examples_to_features, batched=True)

Map: 100%|██████████| 819/819 [00:00<00:00, 7442.98 examples/s]


In [12]:
dataset_samsum_pt["train"]

Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 14731
})

In [13]:
dataset_samsum_pt["train"]["input_ids"][1]

[533,
 2008,
 13613,
 171,
 5337,
 79,
 197,
 5466,
 33,
 22,
 32,
 1292,
 685,
 16907,
 171,
 9308,
 10,
 57,
 1237,
 35,
 533,
 2008,
 13613,
 171,
 3926,
 700,
 27925,
 16907,
 171,
 5446,
 1]

In [14]:
dataset_samsum_pt["train"]["attention_mask"][1]

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1]

In [15]:
# Training Section

#DataCollector divide large amount of data as batch based on your memory
from transformers import DataCollatorForSeq2Seq

seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

In [16]:
from transformers import TrainingArguments, Trainer

trainer_args = TrainingArguments(
    output_dir="pegasus-samsum", num_train_epochs=1, warmup_steps=500,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    weight_decay=0.01, logging_steps=10,
    eval_strategy="steps", eval_steps=500, save_steps=1e6,
    gradient_accumulation_steps=16
)

In [17]:
trainer = Trainer(model=model_pegasus, args=trainer_args,
                  processing_class=tokenizer,
                  data_collator=seq2seq_data_collator,
                  train_dataset=dataset_samsum_pt["test"],
                  # In here im using test data as training data because of practice but in realtime use train data rather test data like dataset_samsum_pt['train]
                  eval_dataset=dataset_samsum_pt["validation"]
                  )

In [18]:
trainer.train()

C:\Users\MSI\anaconda3\envs\GEN_AI_ENV\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
52,977.628906,63.671104


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.75it/s]


TrainOutput(global_step=52, training_loss=1010.3124788724459, metrics={'train_runtime': 1091.6365, 'train_samples_per_second': 0.75, 'train_steps_per_second': 0.048, 'total_flos': 20930926804992.0, 'train_loss': 1010.3124788724459, 'epoch': 1.0})

In [22]:
# Evaluation

def generate_batch_size_chunks(list_of_elements, batch_size):
    #Split dataset into small batches
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i + batch_size]

def calculate_metric_on_test_ds(dataset, metric, model, tokenizer, batch_size=16,
                               device=device,
                               column_text='article',
                               column_summary='highlights'):
    article_batches = list(generate_batch_size_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_size_chunks(dataset[column_summary], batch_size))

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches), total=len(article_batches)):

        inputs = tokenizer(article_batch,max_length=1024,truncation=True,padding="max_length", return_tensors="pt")

        summaries = model.generate(input_ids = inputs["input_ids"].to(device),
                                  attention_mask=inputs["attention_mask"].to(device),
                                  length_penalty=0.8, num_beams=8, max_length=128)


        #Finally, we decode the generated text and replace the token and add the decoded texts with  references to the metrixs.
        decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True,
                                             clean_up_tokenization_spaces=True)
                            for s in summaries]

        decoded_summaries = [d.replace("", " ") for d in decoded_summaries]

        metric.add_batch(predictions=decoded_summaries, references=target_batch)

        #Finally return compute and ROUGE scores.
        score = metric.compute()
        return score

        
    



    

In [24]:
rouge_names = ['rouge1','rouge2','rougeL','rougeLsum']

rouge_metric = evaluate.load('rouge')

In [26]:
score = calculate_metric_on_test_ds(
    dataset_samsum['test'][0:10], rouge_metric, trainer.model, tokenizer, batch_size = 2, column_text = 'dialogue', column_summary= 'summary'
)

rouge_dict = dict((rn, score[rn]) for rn in rouge_names )

pd.DataFrame(rouge_dict, index = [f'pegasus'] )

  0%|          | 0/5 [00:06<?, ?it/s]


,rouge1,rouge2,rougeL,rougeLsum
pegasus,0.02976,0.0,0.02976,0.02976


In [40]:
trainer.save_model("pegasus-samsum-model")
tokenizer.save_pretrained("pegasus-samsum-model")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.08it/s]


('pegasus-samsum-model\\tokenizer_config.json',
 'pegasus-samsum-model\\tokenizer.json')

In [56]:
#Prediction

gen_kwargs = {"length_penalty": 0.8, "num_beams":2, "max_length": 64}



sample_text = dataset_samsum["test"][0]["dialogue"]

reference = dataset_samsum["test"][0]["summary"]

pipe = pipe = pipeline(
    "feature-extraction",
    model="./pegasus-samsum-model",
    tokenizer="./pegasus-samsum-model",  # must point to the same folder
    device=-1  # use CPU
)


print("Dialogue:")
print(sample_text)


print("\nReference Summary:")
print(reference)




Loading weights: 100%|██████████| 131/131 [00:00<00:00, 5954.33it/s]

Dialogue:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Reference Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.
